In [22]:
import json
import random
random.seed(42)
model_name = "0209_modernbert_margin_small_wo_white"
date = model_name.split("_")[0]
with open(f"npm_inference_results/{date}/{model_name}_cls_full.jsonl", "r") as f:
    data_full = [json.loads(line) for line in f]
    data = [d for d in data_full if d["labels"]!=[]]
with open(f"npm_inference_results/{date}/{model_name}_predword.jsonl", "r") as f:
    test_data = [json.loads(line) for line in f]
#with open(f"npm_inference_results/{model_name}.jsonl", "r") as f:
#    test_data_full = [json.loads(line) for line in f]

In [15]:
import json
import random
random.seed(42)

model_name = "answerdotai_ModernBERT-large"
with open(f"npm_inference_results/{model_name}_dev.jsonl", "r") as f:
    data_full = [json.loads(line) for line in f]
    data = [d for d in data_full if d["labels"]!=[]]
with open(f"npm_inference_results/{model_name}_predword.jsonl", "r") as f:
    test_data = [json.loads(line) for line in f]

In [ ]:
len(data), len(test_data), len(data_full)

(325, 364, 1000)

In [12]:
print(data[0].keys())
print(data[0]["spans"][0].keys())

dict_keys(['id_name', 'source_id', 'task_type', 'model', 'source', 'source_info', 'response', 'labels', 'split', 'spans'])
dict_keys(['text', 'start', 'end', 'labels', 'labels_indexed', 'sentence_index', 'token_span', 'top_k_scores', 'top_k_sims', 'max_score', 'max_sim'])


In [2]:
data[0]["spans"][0]

{'text': 'Based',
 'start': 0,
 'end': 5,
 'labels': ['ARGM-ADV', 'V'],
 'labels_indexed': ['0-V', '4-ARGM-ADV'],
 'sentence_index': 0,
 'token_span': [0, 1],
 'top_k_scores': [3.3769941329956055,
  3.261486053466797,
  3.246997833251953,
  3.2287755012512207,
  3.2271268367767334],
 'top_k_sims': [0.3915821611881256,
  0.3117568790912628,
  0.4153342843055725,
  0.4123404622077942,
  0.43544310331344604],
 'max_score': 3.3769941329956055,
 'max_sim': 0.43544310331344604}

In [2]:
def annotate_spans_with_labels(item, include_types=None, mode="any", iou_thresh=0.5):
    """
    item: {
      'labels': [{'start': int, 'end': int, 'label_type': str, ...}, ...],
      'spans':  [{'start': int, 'end': int, ...}, ...]
    }
    include_types: 例 ['Evident Conflict'] にすると、その種類のラベルだけを真とみなす
    mode: 'any'（部分一致でOK） / 'iou'（IoUでしきい値判定）
    iou_thresh: mode='iou' のときの閾値（例 0.5）
    """

    def overlap_len(a0, a1, b0, b1):
        # start: inclusive, end: exclusive 前提
        return max(0, min(a1, b1) - max(a0, b0))

    def iou(a0, a1, b0, b1):
        inter = overlap_len(a0, a1, b0, b1)
        if inter == 0:
            return 0.0
        union = (a1 - a0) + (b1 - b0) - inter
        return inter / union if union > 0 else 0.0

    labels = item.get("labels", [])
    spans = item.get("spans", [])

    # 使うラベルだけ残す
    if include_types is not None:
        labels = [L for L in labels if L.get("label_type") in include_types]

    for sp in spans:
        s0, s1 = sp["start"], sp["end"]
        hit = False
        for L in labels:
            l0, l1 = L["start"], L["end"]
            if mode == "any":
                if overlap_len(s0, s1, l0, l1) > 0:
                    hit = True
                    break
            elif mode == "iou":
                if iou(s0, s1, l0, l1) >= iou_thresh:
                    hit = True
                    break
        sp["hallucinated"] = 1 if hit else 0

    return item

### spanごとに判断

In [23]:
import pandas as pd
import numpy as np

data = [annotate_spans_with_labels(d) for d in data]
test_data = [annotate_spans_with_labels(d) for d in test_data]


def make_features(data):
    span_data = []
    for d in data:
        #mean_sim = sum(sp["max_sim"] for sp in d["spans"]) / len(d["spans"]) if len(d["spans"]) > 0 else 0.0
        for sp in d["spans"]:
            span_data.append({
                "id": d["id_name"],
                "task_type": d["task_type"],
                "text": sp["text"],
                "tag": sp["labels"],
                "start": sp["start"],
                "end": sp["end"],
                "max_score": sp["max_score"],
                "1st_score": sp["top_k_scores"][np.argmax(sp["top_k_sims"])] if len(sp["top_k_scores"]) > 0 else 0.0,
                "2nd_score": sp["top_k_scores"][1] if len(sp["top_k_scores"]) > 1 else 0.0,
                "max_sim": sp["max_sim"],
                "1st_sim": sp["top_k_sims"][0] if len(sp["top_k_sims"]) > 0 else 0.0,
                "2nd_sim": sorted(sp["top_k_sims"], reverse=True)[1] if len(sp["top_k_sims"]) > 1 else 0.0,
                "sim_score": max([sc * si for sc, si in zip(sp["top_k_scores"], sp["top_k_sims"])]),
                "token_len": sp["token_span"][1] - sp["token_span"][0],
                "char_len": sp["end"] - sp["start"],
                "sentence_index": sp["sentence_index"],
                "label": sp["hallucinated"]
            })
    return span_data

span_data = make_features(data)
span_data_test = make_features(test_data)

        
span_df = pd.DataFrame(span_data)
span_df_test = pd.DataFrame(span_data_test)
span_df_test.head(5)

,id,task_type,text,tag,start,end,max_score,1st_score,2nd_score,max_sim,1st_sim,2nd_sim,sim_score,token_len,char_len,sentence_index,label
0,28,Summary,Three women,"[ARG0, ARG1, ARG2]",0,11,3.504336,3.504336,3.179654,0.660846,0.660846,0.640274,2.315827,2,11,0,1
1,28,Summary,including,"[ARG0, ARG1, V]",13,22,2.755082,2.728104,2.746591,0.324190,0.292108,0.292108,0.884425,1,9,0,1
2,28,Summary,Keonna Thomas of Philadelphia,"[ARG0, ARG1]",23,52,3.320057,3.320057,3.139681,0.590398,0.590398,0.535369,1.960154,4,29,0,1
3,28,Summary,were charged,[V],54,66,4.162865,3.969503,3.969503,0.688180,0.677872,0.677872,2.821888,2,12,0,1
4,28,Summary,with attempting,"[ARG2, V]",67,82,3.766014,3.766014,3.453777,0.698058,0.698058,0.620238,2.628896,2,15,0,1


In [4]:
def eval_char(item,hal_type=None):
    hal_label = []
    for d in item["labels"]:
        if hal_type is None or d.get("label_type") == hal_type:
            hal_label.extend(list(range(d["start"], d["end"])))
    hal_label = set(hal_label)

    hal_pred = []
    for sp in item["spans"]:
        if sp["predicted"] == 1:
            hal_pred.extend(list(range(sp["start"], sp["end"])))
    hal_pred = set(hal_pred)
    tp = len(hal_label & hal_pred)
    fp = len(hal_pred - hal_label)
    fn = len(hal_label - hal_pred)
    # print(hal_label, hal_pred)
    return tp, fp, fn

def compute_metrics_per_char(result_data, hal_type=None):
    tp = fp = fn = 0
    for item in result_data:
        tp_i, fp_i, fn_i = eval_char(item, hal_type=hal_type)
        tp += tp_i
        fp += fp_i
        fn += fn_i
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return {
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "Precision": f"{precision*100:.2f}",
        "Recall": f"{recall*100:.2f}",
        "F1": f"{f1*100:.2f}",
    }

In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler

def classifier(train_data, test_data, features, thr=0.5):
    train_data = [annotate_spans_with_labels(d) for d in train_data]
    test_data = [annotate_spans_with_labels(d) for d in test_data]
    span_train_data = make_features(train_data)
    span_test_data = make_features(test_data)
    span_df = pd.DataFrame(span_train_data)
    span_df_test = pd.DataFrame(span_test_data)

    X = span_df[features]
    y = span_df["label"]
    X_test = span_df_test[features]
    y_test = span_df_test["label"]

    X_train, X_dev, y_train, y_dev = train_test_split(
        X, y, test_size=None, random_state=42, stratify=y
    )

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    #X_dev = scaler.transform(X_dev)
    X_test = scaler.transform(X_test)

    clf = LogisticRegression(max_iter=1000, class_weight="balanced")
    clf.fit(X_train, y_train)

    y_probs = clf.predict_proba(X_test)[:, 1]
    threshold = thr
    y_pred = (y_probs >= threshold).astype(int)
    #y_pred = clf.predict(X_test)
    print("positive:", sum(y_pred), "out of", len(y_pred))
    print(classification_report(y_test, y_pred, digits=3))
    
    span_num_list = [len(d["spans"]) for d in test_data]
    for n in range(len(test_data)):
        for i in range(span_num_list[n]):
            test_data[n]["spans"][i]["predicted"] = int(y_pred[sum(span_num_list[:n]) + i])
            test_data[n]["spans"][i]["clf_score"] = float(y_probs[sum(span_num_list[:n]) + i])
    print("ALL:", compute_metrics_per_char(test_data))
    print("QA:", compute_metrics_per_char([d for d in test_data if d["task_type"] == "QA"]))
    print("Summary:", compute_metrics_per_char([d for d in test_data if d["task_type"] == "Summary"]))
    coef = clf.coef_[0]  # shape: (n_features,)
    for name, c in zip(features, coef):
        print(f"{name:15s}  coef = {c:.4f}")
    return test_data

In [24]:
features = ["max_score", "max_sim", "token_len","start","end", "sentence_index"]
test_data = classifier(data_full,test_data, features)

positive: 7321 out of 19170
              precision    recall  f1-score   support

           0      0.882     0.664     0.758     15737
           1      0.278     0.592     0.378      3433

    accuracy                          0.651     19170
   macro avg      0.580     0.628     0.568     19170
weighted avg      0.774     0.651     0.690     19170

ALL: {'TP': 27427, 'FP': 71570, 'FN': 21899, 'Precision': '27.70', 'Recall': '55.60', 'F1': '36.98'}
QA: {'TP': 19094, 'FP': 41425, 'FN': 12241, 'Precision': '31.55', 'Recall': '60.94', 'F1': '41.57'}
Summary: {'TP': 8333, 'FP': 30145, 'FN': 9658, 'Precision': '21.66', 'Recall': '46.32', 'F1': '29.51'}
max_score        coef = -0.0471
max_sim          coef = -0.6129
token_len        coef = 0.0635
start            coef = 0.1020
end              coef = 0.0994
sentence_index   coef = 0.3861


In [10]:
# タイプ別のrecall
hal_types = ["Evident Conflict", "Subtle Conflict", "Evident Baseless Info", "Subtle Baseless Info"]
for hal_type in hal_types:
    print(f"Type: {hal_type}")
    qa_metrics = compute_metrics_per_char([d for d in test_data if d["task_type"] == "QA"], hal_type=hal_type)
    sum_metrics = compute_metrics_per_char([d for d in test_data if d["task_type"] == "Summary"], hal_type=hal_type)
    all_metrics = compute_metrics_per_char(test_data, hal_type=hal_type)
    print("ALL:", "Num:", all_metrics["TP"] + all_metrics["FN"], "Recall:", all_metrics["Recall"])
    print("QA:", "Num:", qa_metrics["TP"] + qa_metrics["FN"], "Recall:", qa_metrics["Recall"])
    print("Summary:", "Num:", sum_metrics["TP"] + sum_metrics["FN"], "Recall:", sum_metrics["Recall"])
    print()

Type: Evident Conflict
ALL: Num: 8161 Recall: 31.99
QA: Num: 2349 Recall: 35.76
Summary: Num: 5812 Recall: 30.47

Type: Subtle Conflict
ALL: Num: 339 Recall: 39.53
QA: Num: 0 Recall: 0.00
Summary: Num: 339 Recall: 39.53

Type: Evident Baseless Info
ALL: Num: 34261 Recall: 64.96
QA: Num: 23441 Recall: 67.95
Summary: Num: 10820 Recall: 58.49

Type: Subtle Baseless Info
ALL: Num: 6565 Recall: 66.76
QA: Num: 5545 Recall: 64.78
Summary: Num: 1020 Recall: 77.55



In [73]:
with open(f"npm_inference_results/{model_name}_predword_clf.jsonl", "w") as f:
    for d in test_data:
        f.write(json.dumps(d, ensure_ascii=False) + "\n")